# 59 — Quantization & distillation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/059-quantization-and-distillation.ipynb)

Compression preserves behaviour, not weights. Here you quantize a small tensor to INT8, measure the rounding error, then soften a teacher's logits with temperature to see what a student can learn from them.

## 1. Quantize and restore

The scale is `max|w| / 127`. Each weight becomes a code, and the restored value is the code times the scale. Compare them row by row.

In [ ]:
import numpy as np                                    # arrays, preinstalled on Colab

w = np.array([-0.9, -0.3, 0.0, 0.45, 1.2])            # one small tensor
scale = np.abs(w).max() / 127                         # symmetric scale
codes = np.clip(np.round(w / scale), -127, 127)       # integer codes
restored = codes * scale                              # dequantized values
error = np.abs(w - restored)                          # absolute error
for original, q, r, e in zip(w, codes, restored, error):   # one row per weight
    print(f"w={original:5.2f} q={q:4.0f} restored={r:7.4f} error={e:.4f}")
print("scale =", round(scale, 5), "bound =", round(scale / 2, 5))   # 0.0047
print("max error =", round(error.max(), 4))           # inside the bound

## 2. Temperature softens the teacher

A teacher's logits are turned into probabilities with softmax. Dividing by `T` before the softmax flattens the distribution and exposes how the teacher ranks the near-miss classes.

In [ ]:
def softmax(z, T):                                    # logits and temperature
    scaled = np.array(z) / T                          # divide before exponentiating
    e = np.exp(scaled - scaled.max())                 # subtract max for stability
    return e / e.sum()                                # normalise to probabilities

logits = [4.0, 1.0, 0.5]                              # one teacher prediction
for T in (1, 2, 5):                                   # three temperatures
    p = softmax(logits, T)                            # softened targets
    print(f"T={T}: {np.round(p, 3)}")                # watch the tail grow

## Your turn

1. Add a weight of 10.0 to the tensor. Recompute the scale and the error on 0.45 — why does per-channel quantization help here?
2. At `T=5`, what are the three probabilities, and which class pair becomes easiest to tell apart for the student?
3. Explain why the teacher's argmax does not change with temperature, while the student's training signal does.